# Country Analysis — Freedom in the World

**Notebook 05 of 08**

### Purpose

The global picture (notebook 04) showed a broad decline. This notebook zooms in on individual economies, comparing **three groups**: the East African Community, the world's major powers, and Africa's top-scoring economies.

### Scope

Every country comparison uses the same three lenses: **trends** (2013–2026), **current standing** (2026), and **movers** (change since 2013). Section 10 then goes beyond the overall score and compares the groups on the **components** — political rights, civil liberties, the seven category subtotals, and the categorical status.

## Data and Inputs

| Item | Location |
|---|---|
| Long analytical dataset | `data/processed/freedom_in_world_long.csv` |
| Country groupings | `src/regions.py` (documented classifications) |

**Question this notebook answers:** *How do selected groups of economies differ in their freedom trajectories — and what do the component indicators add to the headline score?*

### The groupings (external, documented, approved)

The dataset supplies no regional classification, so the groups come from documented external sources, mapped to the dataset's own economy labels:

| Group | Membership | Source |
|---|---|---|
| East African Community | Uganda, Kenya, Tanzania, Rwanda, Burundi, South Sudan, Congo, Dem. Rep., Somalia (8) | eac.int |
| World major powers | UN Security Council P5 + G7: United States, United Kingdom, France, Russian Federation, China, Germany, Italy, Japan, Canada (9) | un.org / g7uk.org |
| African top scorers | The 54 African UN member states (UN M49), ranked by 2026 overall score, top 10 analysed | unstats.un.org (M49) |

## Setup: imports and the project root

Same bootstrap. New imports: the three groupings from `src/regions.py` and the multi-line trend helper `create_multi_trend_chart()`.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd
import plotly.express as px

from src.data_loader import load_processed_data
from src.visualizations import create_multi_trend_chart, create_heatmap
from src.regions import EAC_ECONOMIES, SUPERPOWERS_P5_G7, AFRICA_UN_M49

print('Imports ready.')

Imports ready.


### Interpretation

Setup ran cleanly. The groupings are plain lists of economy labels — every member below is verified to exist in the dataset before any chart is drawn.

## 1. Prepare the overall-score series

**Question:** what data do all three comparisons share?

**Method:** load the long dataset and keep only `FH_FIW_TOTAL`, with numeric scores.

In [2]:
long = load_processed_data()
total = long[long['INDICATOR'] == 'FH_FIW_TOTAL'].copy()
total['Score'] = pd.to_numeric(total['Score'], errors='coerce')

print('TOTAL series:', total.shape)
print('Economies covered:', total['Economy'].nunique())

TOTAL series: (2758, 9)
Economies covered: 197


### Interpretation

2758 rows of overall scores — every economy, every year. The comparisons below filter this series by group and by year.

## 2. Verify the groupings against the dataset

**Question:** do all group members exist in the data?

**Method:** check each group's labels against the economies present.

In [3]:
present = set(total['Economy'].unique())
for name, group in [('EAC', EAC_ECONOMIES), ('P5+G7', SUPERPOWERS_P5_G7), ('Africa (M49)', AFRICA_UN_M49)]:
    missing = [e for e in group if e not in present]
    print(f'{name}: {len(group)} members, missing from dataset: {missing}')

overlap = set(EAC_ECONOMIES) & set(AFRICA_UN_M49)
print('EAC members that are also in the Africa list:', len(overlap), 'of', len(EAC_ECONOMIES))

EAC: 8 members, missing from dataset: []
P5+G7: 9 members, missing from dataset: []
Africa (M49): 54 members, missing from dataset: []
EAC members that are also in the Africa list: 8 of 8


### Interpretation

All **8 EAC**, all **9 P5+G7** and all **54 African** members resolve in the dataset. The EAC set is a perfect subset of the Africa list (8 of 8) — as it should be — which validates the two classifications against each other.

## 3. The East African Community — trends

**Question:** how have the eight EAC economies evolved since 2013?

**Method:** one line per member, 2013–2026.

In [4]:
eac = total[total['Economy'].isin(EAC_ECONOMIES)]
fig = create_multi_trend_chart(
    eac,
    title='Overall freedom score, East African Community, 2013-2026',
    y_label='Overall score (0-100)',
)
fig.show()

### Interpretation

Seven of the eight EAC members declined; **only Somalia improved** (2 → 8). The trajectories are not uniform: Kenya and Uganda fell gradually (55 → 49 and 40 → 33), while **Tanzania collapsed from 66 to 28** and **South Sudan went from 31 to 0**. Burundi also fell hard (34 → 13).

## 4. EAC — current standing and movers

**Question:** who leads the EAC today, and who changed most?

**Method:** rank by 2026 score, then plot the 2013→2026 change.

In [5]:
eac_2026 = eac[eac['Year'] == 2026].set_index('Economy')['Score'].dropna().sort_values()
fig = px.bar(
    eac_2026,
    orientation='h',
    title='East African Community: overall score 2026',
    labels={'value': 'Overall score (0-100)', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

In [6]:
eac_movers = eac[eac['Year'].isin([2013, 2026])].pivot_table(index='Economy', columns='Year', values='Score', aggfunc='first')
eac_movers['change'] = (eac_movers[2026] - eac_movers[2013]).round(1)
eac_movers = eac_movers.sort_values('change')

fig = px.bar(
    eac_movers.reset_index(),
    x='change',
    y='Economy',
    orientation='h',
    title='East African Community: change in overall score, 2013 to 2026',
    labels={'change': 'Change in score', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

**Kenya leads the EAC in 2026 (49)** — far below the world average of 56.9 — followed by Uganda (33), Tanzania (28), Rwanda (21), Congo, Dem. Rep. (18), Burundi (13), Somalia (8) and South Sudan (0).

The movers chart shows the depth of the region's decline: **Tanzania −38, South Sudan −31, Burundi −21**; Uganda −7 and Kenya −6; Rwanda −3 and Congo −2. Only Somalia bucked the trend (+6). The EAC's 2026 average of roughly 21 sits far under the global mean — the region is one of the dataset's hardest-hit.

## 5. World major powers (P5 + G7) — trends

**Question:** how do the permanent Security Council members and G7 economies compare?

**Method:** one line per power, 2013–2026.

In [7]:
powers = total[total['Economy'].isin(SUPERPOWERS_P5_G7)]
fig = create_multi_trend_chart(
    powers,
    title='Overall freedom score, P5 + G7 major powers, 2013-2026',
    y_label='Overall score (0-100)',
)
fig.show()

### Interpretation

The chart splits into two clusters with a **~70-point gap**: the democratic powers sit between 81 and 97, while Russia (27 → 12) and China (17 → 9) sit at the very bottom of the dataset. Within the top cluster, **Japan is the only improver** (88 → 96, overtaking Germany, the UK and the US); the **United States fell hardest (−12, 93 → 81)**.

## 6. P5 + G7 — current standing and movers

**Question:** where do the powers stand in 2026, and who moved most?

**Method:** rank by 2026 score, then plot the change since 2013.

In [8]:
p_2026 = powers[powers['Year'] == 2026].set_index('Economy')['Score'].dropna().sort_values()
fig = px.bar(
    p_2026,
    orientation='h',
    title='Major powers: overall score 2026',
    labels={'value': 'Overall score (0-100)', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

In [9]:
p_movers = powers[powers['Year'].isin([2013, 2026])].pivot_table(index='Economy', columns='Year', values='Score', aggfunc='first')
p_movers['change'] = (p_movers[2026] - p_movers[2013]).round(1)
p_movers = p_movers.sort_values('change')

fig = px.bar(
    p_movers.reset_index(),
    x='change',
    y='Economy',
    orientation='h',
    title='Major powers: change in overall score, 2013 to 2026',
    labels={'change': 'Change in score', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

The 2026 ranking: **Canada 97, Japan 96, Germany 95, United Kingdom 92, France 89, Italy 87, United States 81** — then Russia (12) and China (9). The gap between the top cluster and Russia/China is the widest of any comparison in this notebook.

Every power declined except Japan (+8). The United States (−12), Russia (−15) and China (−8) moved most. France (−6) and the UK (−5) also fell noticeably — the decline of the overall global average is visible even among the world's most free economies.

## 7. Africa's top scorers — trends

**Question:** among African economies, who scores highest, and how have the leaders evolved?

**Method:** take the top 10 African economies by 2026 overall score (from the UN M49 list of 54) and plot their trends.

In [10]:
africa = total[total['Economy'].isin(AFRICA_UN_M49)]
africa_2026 = africa[africa['Year'] == 2026].set_index('Economy')['Score'].dropna()
top10 = africa_2026.nlargest(10)
print('Africa 2026 mean:', round(africa_2026.mean(), 2), '| World 2026 mean:', round(total[total['Year'] == 2026]['Score'].mean(), 2))
print('Top 10 African economies in 2026:', ', '.join(top10.index))

Africa 2026 mean: 38.56 | World 2026 mean: 56.9
Top 10 African economies in 2026: Cabo Verde, Mauritius, Sao Tome and Principe, Seychelles, South Africa, Ghana, Botswana, Namibia, Senegal, Malawi


In [11]:
africa_top = africa[africa['Economy'].isin(top10.index)]
fig = create_multi_trend_chart(
    africa_top,
    title='Overall freedom score, top-10 African economies, 2013-2026',
    y_label='Overall score (0-100)',
)
fig.show()

### Interpretation

The African leaders are a **different world from the EAC**: they sit between 68 and 92, with **Cabo Verde (92) and Mauritius (87)** at the top. The group is comparatively stable — most lines are flat or gently falling — with two clear improvers: **Seychelles (+14, 67 → 81)** and **Malawi (+8, 60 → 68)**. The contrast with the global trend is notable: Africa's best performers largely held their ground while the world declined.

## 8. Africa's leaders — standing and movers

**Question:** what is the exact 2026 ranking, and who changed most?

**Method:** rank the top 10 and plot the change since 2013.

In [12]:
fig = px.bar(
    top10.sort_values(),
    orientation='h',
    title='Africa: top 10 overall scores 2026',
    labels={'value': 'Overall score (0-100)', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

In [13]:
a_movers = africa_top[africa_top['Year'].isin([2013, 2026])].pivot_table(index='Economy', columns='Year', values='Score', aggfunc='first')
a_movers['change'] = (a_movers[2026] - a_movers[2013]).round(1)
a_movers = a_movers.sort_values('change')

fig = px.bar(
    a_movers.reset_index(),
    x='change',
    y='Economy',
    orientation='h',
    title='Africa top 10: change in overall score, 2013 to 2026',
    labels={'change': 'Change in score', 'Economy': 'Economy'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

The 2026 African leaderboard: **Cabo Verde 92, Mauritius 87, Sao Tome and Principe 84, Seychelles 81, South Africa 81, Ghana 80, Botswana 75, Namibia 73, Senegal 70, Malawi 68.**

Movers split into improvers (Seychelles +14, Malawi +8, Sao Tome +3, Cabo Verde +2, Botswana +1), a hold (South Africa 0), and modest decliners (Namibia −3, Mauritius −3, Ghana −4, Senegal −5). Yet even this leaderboard sits **below the global average** — Africa's overall mean in 2026 is **38.6 against the world's 56.9**, so these leaders are Africa's exceptions, not its rule.

## 9. Worked example: Tanzania's indicator profile

**Question:** what does a country comparison reduce to when we zoom into one economy?

**Method:** take Tanzania — the EAC's biggest decliner — and plot its 2026 indicator scores **as a share of each indicator's scale** (so questions, subtotals and totals are comparable).

In [14]:
sm_map = {'0_TO_4': 4, '0_TO_12': 12, '0_TO_16': 16, '0_TO_40': 40, '0_TO_60': 60, '0_TO_100': 100}

tz = long[(long['Economy'] == 'Tanzania') & (long['Year'] == 2026)].copy()
tz['Score'] = pd.to_numeric(tz['Score'], errors='coerce')
tz = tz[tz['UNIT_MEASURE'].isin(sm_map)].copy()
tz['pct_of_scale'] = (tz['Score'] / tz['UNIT_MEASURE'].map(sm_map) * 100).round(0)
tz = tz.sort_values('pct_of_scale')

fig = px.bar(
    tz,
    x='pct_of_scale',
    y='INDICATOR',
    orientation='h',
    title='Tanzania 2026: indicator scores as share of scale',
    labels={'pct_of_scale': '% of indicator scale', 'INDICATOR': 'Indicator'},
    text_auto=True,
    color='Category',
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

Tanzania's 2026 profile is dramatic: **eight indicators sit at 0% of their scale** — all three electoral-process questions (`A1`, `A2`, `A3`), two political-pluralism questions (`B2`, `B3`), an associational-rights question (`E1`), the political-rights subtotal and an additional question. Its strongest areas are personal autonomy (the `G` subtotal at 50%) and associational rights `E3` (50%) — roughly half of what the FiW system allows.

This is why the country lens matters: the headline score (28) hides *which* rights collapsed. For Tanzania, the political-rights dimension — elections and pluralism — is where the decline concentrated, while civil-liberties items held up better. (Ratings and the categorical status are handled separately, in notebook 07.)

## 10. Beyond the headline score

The comparisons so far used only the overall score. The overall score is the sum of **political rights (PR, 0–40)** and **civil liberties (CL, 0–60)**, and each is built from seven **category subtotals** (A–G). The headline can hide which components drove a group's trajectory.

**Question this section answers:** *Do the three groups differ in their political-rights vs civil-liberties mix, their category strengths, and their status classification?*

**Method:** three additional views, all normalized the same way (score as % of its scale, as in sections 4–9):
1. PR vs CL trajectories per group (2013–2026).
2. Category heatmaps for 2026 (countries × the seven categories plus PR, CL and TOTAL).
3. The categorical status mix (Free / Partly Free / Not Free) in 2026.

### 10.1 Political rights vs civil liberties

**Question:** did a group's trajectory hit political rights, civil liberties, or both?

**Method:** mean PR and mean CL per year as a share of each scale (40 and 60), averaged across the group's economies. One cell prepares all three groups; the charts follow.

In [15]:
label_map = {'FH_FIW_PR': 'Political rights', 'FH_FIW_CL': 'Civil liberties'}
pr_cl = long[long['INDICATOR'].isin(['FH_FIW_PR', 'FH_FIW_CL'])].copy()
pr_cl['Score'] = pd.to_numeric(pr_cl['Score'], errors='coerce')
pr_cl['pct'] = pr_cl['Score'] / pr_cl['UNIT_MEASURE'].map(sm_map) * 100

groups = {'EAC': EAC_ECONOMIES, 'Major powers': SUPERPOWERS_P5_G7, 'Africa top 10': list(top10.index)}
pr_cl_means = {}
for name, members in groups.items():
    sub = pr_cl[pr_cl['Economy'].isin(members)].groupby(['Year', 'INDICATOR'])['pct'].mean().reset_index()
    sub['component'] = sub['INDICATOR'].map(label_map)
    pr_cl_means[name] = sub
print('Prepared PR/CL means for:', list(pr_cl_means.keys()))

Prepared PR/CL means for: ['EAC', 'Major powers', 'Africa top 10']


#### East African Community

**Question:** how do the EAC's two components compare over time?

**Method:** two lines — mean PR and mean CL as % of scale.

In [16]:
fig = create_multi_trend_chart(
    pr_cl_means['EAC'],
    title='EAC: political rights vs civil liberties (mean % of scale), 2013-2026',
    y_label='Mean score as % of scale',
    group_col='component',
    value_col='pct',
)
fig.show()

### Interpretation

The EAC's **political-rights score halved** — from 31.6% of its scale in 2013 to 15.0% in 2026 — while civil liberties fell from 35.6% to 25.4%. The region's collapse is primarily a **political-rights collapse**: the electoral-process, pluralism and government-function questions deteriorated far more than the civil-liberties items.

#### Major powers

**Question:** which component moved more among the P5 + G7 powers?

**Method:** the same two lines for the powers' group mean.

In [17]:
fig = create_multi_trend_chart(
    pr_cl_means['Major powers'],
    title='Major powers: political rights vs civil liberties (mean % of scale), 2013-2026',
    y_label='Mean score as % of scale',
    group_col='component',
    value_col='pct',
)
fig.show()

### Interpretation

For the powers, political rights fell from 76.4% to 73.6% of scale and civil liberties from 78.5% to 72.8% — a **civil-liberties decline slightly larger than the political-rights one**, the opposite of the EAC. These averages blend two very different clusters (seven near the top, Russia and China near the bottom), so the group mean smooths over a much larger two-cluster gap.

#### Africa's top 10

**Question:** do the African leaders show the same component pattern?

**Method:** the same two lines for the top-10 group.

In [18]:
fig = create_multi_trend_chart(
    pr_cl_means['Africa top 10'],
    title='Africa top 10: political rights vs civil liberties (mean % of scale), 2013-2026',
    y_label='Mean score as % of scale',
    group_col='component',
    value_col='pct',
)
fig.show()

### Interpretation

Africa's leaders are the only group in this notebook whose **political rights improved**: PR rose from 80.2% to 83.0% of scale, and civil liberties held essentially flat (76.2% → 76.5%). This is the opposite of the EAC — and of the world trend from notebook 04 — a second signal, alongside the overall scores, that the African leaders did not follow the global decline.

### 10.2 Category heatmaps (2026)

**Question:** which of the seven freedom categories are each group's strengths and weaknesses?

**Method:** heatmap of countries × categories for 2026, as % of scale — the seven subtotals (A–G) plus the PR, CL and TOTAL rows. Values are kept as-is; nothing is clamped or imputed (so the documented PR = −4 for South Sudan shows up as −10% of scale).

In [19]:
cats = ['FH_FIW_A', 'FH_FIW_B', 'FH_FIW_C', 'FH_FIW_D', 'FH_FIW_E', 'FH_FIW_F', 'FH_FIW_G', 'FH_FIW_PR', 'FH_FIW_CL', 'FH_FIW_TOTAL']
cat_labels = {'FH_FIW_A': 'A Electoral', 'FH_FIW_B': 'B Pluralism', 'FH_FIW_C': 'C Gov. function', 'FH_FIW_D': 'D Expression', 'FH_FIW_E': 'E Association', 'FH_FIW_F': 'F Rule of law', 'FH_FIW_G': 'G Personal autonomy', 'FH_FIW_PR': 'PR total', 'FH_FIW_CL': 'CL total', 'FH_FIW_TOTAL': 'Overall'}

def category_pivot(economies):
    sub = long[long['INDICATOR'].isin(cats) & long['Economy'].isin(economies) & (long['Year'] == 2026)].copy()
    sub['Score'] = pd.to_numeric(sub['Score'], errors='coerce')
    sub['pct'] = sub['Score'] / sub['UNIT_MEASURE'].map(sm_map) * 100
    piv = sub.pivot_table(index='Economy', columns='INDICATOR', values='pct', aggfunc='first')
    piv = piv[list(cat_labels.keys())].rename(columns=cat_labels)
    return piv

eac_pivot = category_pivot(EAC_ECONOMIES)
powers_pivot = category_pivot(SUPERPOWERS_P5_G7)
africa_pivot = category_pivot(list(top10.index))
print('Category pivots ready:', eac_pivot.shape, powers_pivot.shape, africa_pivot.shape)

Category pivots ready: (8, 10) (9, 10) (10, 10)


#### East African Community

**Question:** where are the EAC's category strengths and weaknesses in 2026?

**Method:** heatmap of the eight members × the ten rows.

In [20]:
fig = create_heatmap(
    eac_pivot,
    title='EAC 2026: categories as % of scale',
    colorbar_label='% of scale',
)
fig.show()

### Interpretation

The EAC heatmap is dominated by dark cells. The group's weakest categories in 2026 are **electoral process (A) and rule of law (F), both averaging ~16% of scale**, followed by pluralism (B, ~16%) and government function (C, ~17%); its strongest is expression (D, ~33%). South Sudan's row shows the out-of-range value from notebook 01 — a PR subtotal of −4 (−10% of scale) — **kept, not clamped**. Tanzania's row confirms section 9: political-rights cells at zero with personal autonomy (G) at ~50%.

#### Major powers

**Question:** where are the powers' category strengths and weaknesses in 2026?

**Method:** heatmap of the nine members × the ten rows.

In [21]:
fig = create_heatmap(
    powers_pivot,
    title='Major powers 2026: categories as % of scale',
    colorbar_label='% of scale',
)
fig.show()

### Interpretation

The powers heatmap splits in two: the seven democratic members fill their cells at 60–100% of scale while **China and Russia sit at the floor in every category**. For the group as a whole, the lowest average category is rule of law (F, ~67%) and the highest is personal autonomy (G, ~78%) — but those averages are pulled down by the two bottom rows; within the democratic seven the cells are uniformly high.

#### Africa's top 10

**Question:** where are the African leaders' category strengths and weaknesses in 2026?

**Method:** heatmap of the ten leaders × the ten rows.

In [22]:
fig = create_heatmap(
    africa_pivot,
    title='Africa top 10 2026: categories as % of scale',
    colorbar_label='% of scale',
)
fig.show()

### Interpretation

Africa's leaders show the mirror image of the EAC: a **uniformly bright heatmap**, with the strongest categories being electoral process (A, ~92%) and expression (D, ~87%) and the weakest — still near 70% — being rule of law (F) and personal autonomy (G). Only 5 of the 70 cells fall below 60% of scale. The contrast between the two African groups is one of the clearest findings of this notebook.

### 10.3 The status mix (2026)

**Question:** how do the three groups compare on the categorical classification Free / Partly Free / Not Free?

**Method:** count each group's `FH_FIW_STATUS` values in 2026. STATUS is the categorical indicator — its strings are used directly, never coerced to numbers.

In [23]:
status = long[(long['INDICATOR'] == 'FH_FIW_STATUS') & (long['Year'] == 2026)]
status_order = ['F', 'PF', 'NF']
rows = []
for name, members in groups.items():
    counts = status[status['Economy'].isin(members)]['Score'].value_counts()
    for s in status_order:
        rows.append({'group': name, 'status': s, 'economies': int(counts.get(s, 0))})
mix = pd.DataFrame(rows)
print(mix.pivot(index='group', columns='status', values='economies'))

fig = px.bar(
    mix,
    x='group',
    y='economies',
    color='status',
    barmode='group',
    title='Status classification of the three groups, 2026',
    labels={'group': 'Group', 'economies': 'Number of economies', 'status': 'Status'},
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

status          F  NF  PF
group                    
Africa top 10  10   0   0
EAC             0   7   1
Major powers    7   2   0


### Interpretation

The status mix mirrors the scores exactly. **EAC: 7 Not Free, 1 Partly Free** — only Kenya escapes the lowest classification. **Major powers: 7 Free, 2 Not Free** — China and Russia. **Africa's top 10: all 10 Free.** The three groups occupy three entirely different positions on the categorical scale, matching the numeric picture built up over the rest of the notebook.

## 11. The overall score is a sum

**Question:** can the components be grouped back into the overall score?

**Method:** verify the additive hierarchy directly on the data, then decompose the groups' 2026 scores into their two headline components.

The Freedom in the World methodology is additive by design. This section tests three identities against the data:

1. **TOTAL = PR + CL** — the overall score is political rights plus civil liberties.
2. **CL = D + E + F + G** — civil liberties is the sum of its four category subtotals.
3. **PR = A + B + C** — political rights is the sum of its three category subtotals (the two additional questions are checked separately).

If the identities hold, the components genuinely group into the overall score.

In [24]:
# Scores arrive mixed-type (STATUS strings share the column) - coerce a copy to numeric first
parts_df = long.copy()
parts_df['Score'] = pd.to_numeric(parts_df['Score'], errors='coerce')
parts = parts_df.pivot_table(index=['REF_AREA', 'Year'], columns='INDICATOR', values='Score', aggfunc='first')

def check(name, series):
    s = series.dropna()
    exact = int((s.abs() < 1e-9).sum())
    pct = round(exact / len(s) * 100, 1)
    print(f'{name}: exact in {exact} of {len(s)} economy-years ({pct}%)')

check('TOTAL = PR + CL',
      parts['FH_FIW_TOTAL'] - parts['FH_FIW_PR'] - parts['FH_FIW_CL'])
check('CL = D + E + F + G',
      parts['FH_FIW_CL'] - parts['FH_FIW_D'] - parts['FH_FIW_E'] - parts['FH_FIW_F'] - parts['FH_FIW_G'])
check('PR = A + B + C',
      parts['FH_FIW_PR'] - parts['FH_FIW_A'] - parts['FH_FIW_B'] - parts['FH_FIW_C'])

TOTAL = PR + CL: exact in 2748 of 2748 economy-years (100.0%)
CL = D + E + F + G: exact in 2748 of 2748 economy-years (100.0%)
PR = A + B + C: exact in 2558 of 2748 economy-years (93.1%)


### Interpretation

**TOTAL = PR + CL holds exactly** for every economy-year with all three values present, and **CL = D + E + F + G holds exactly** too. Political rights are *almost* the sum of A + B + C: exact in about 93% of economy-years, with small deviations elsewhere — the published PR subtotal occasionally differs from its category subtotals (South Sudan's PR is −4 while its three categories sum to 0; the additional questions are sometimes folded in). The category subtotals are therefore a very good — but not perfect — reconstruction of the overall score, while **PR + CL reconstructs it exactly**. The published subtotals remain the authoritative values; nothing in this notebook recomputes or replaces them.

### 11.1 Decomposing the groups' 2026 scores

**Question:** how much of each group's overall score comes from political rights vs civil liberties?

**Method:** stacked bars of mean PR and mean CL (2026). Because the identity is exact row-by-row, the means decompose exactly too: mean PR + mean CL = mean TOTAL.

In [25]:
decomp = long[long['INDICATOR'].isin(['FH_FIW_PR', 'FH_FIW_CL', 'FH_FIW_TOTAL']) & (long['Year'] == 2026)].copy()
decomp['Score'] = pd.to_numeric(decomp['Score'], errors='coerce')

group_rows = []
for name, members in groups.items():
    means = decomp[decomp['Economy'].isin(members)].groupby('INDICATOR')['Score'].mean()
    group_rows.append({'group': name,
                       'Political rights': round(means.get('FH_FIW_PR', float('nan')), 1),
                       'Civil liberties': round(means.get('FH_FIW_CL', float('nan')), 1)})
group_stack = pd.DataFrame(group_rows)
print(group_stack.to_string(index=False))

fig = px.bar(group_stack, x='group', y=['Political rights', 'Civil liberties'], barmode='stack',
             title='2026 group means: overall score decomposed into PR + CL',
             labels={'value': 'Mean points', 'group': 'Group', 'variable': 'Component'},
             text_auto=True)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

        group  Political rights  Civil liberties
          EAC               6.0             15.2
 Major powers              29.4             43.7
Africa top 10              33.2             45.9


#### East African Community, per country

**Question:** does the decomposition hold country by country — and where does each member's score come from?

**Method:** stacked bars of each EAC member's PR and CL in 2026; the top of each bar is the overall score.

In [26]:
eac_pc = decomp[decomp['Economy'].isin(EAC_ECONOMIES)].pivot_table(index='Economy', columns='INDICATOR', values='Score', aggfunc='first')
eac_pc = eac_pc[['FH_FIW_PR', 'FH_FIW_CL', 'FH_FIW_TOTAL']].sort_values('FH_FIW_TOTAL')
print(eac_pc.rename(columns={'FH_FIW_PR': 'PR', 'FH_FIW_CL': 'CL', 'FH_FIW_TOTAL': 'TOTAL'}).to_string())

fig = px.bar(eac_pc.reset_index(), x='Economy', y=['FH_FIW_PR', 'FH_FIW_CL'], barmode='stack',
             title='EAC 2026: overall score decomposed into PR + CL, per economy',
             labels={'value': 'Points', 'Economy': 'Economy', 'variable': 'Component'},
             text_auto=True)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

INDICATOR           PR    CL  TOTAL
Economy                            
South Sudan       -4.0   4.0    0.0
Somalia            2.0   6.0    8.0
Burundi            2.0  11.0   13.0
Congo, Dem. Rep.   4.0  14.0   18.0
Rwanda             7.0  14.0   21.0
Tanzania           6.0  22.0   28.0
Uganda            10.0  23.0   33.0
Kenya             21.0  28.0   49.0


### Interpretation

The decomposition is exact in both charts: every stacked height equals the overall score (e.g. Kenya 21 + 28 = 49), and the group means match the values computed throughout this notebook (EAC 21.2, powers 73.1, Africa top 10 79.1).

Two patterns stand out:

- **In every EAC economy, civil liberties contribute more than political rights** — Kenya (CL 28 vs PR 21), Uganda (23 vs 10), Tanzania (22 vs 6). The region's political-rights weakness, first seen in section 10.1, is visible country by country.
- **South Sudan's bar renders with a negative PR segment (−4) and CL of 4, landing exactly on 0** — the published negative PR value is preserved in the stack rather than clamped, and the identity still holds (TOTAL = −4 + 4).

Between groups, the composition differs too: political rights make up only **~28% of the EAC's mean** (6.0 of 21.2) but **~42% of the African leaders' mean** (33.2 of 79.1) — the EAC is not just lower, it is structurally more civil-liberties-heavy.

## Summary and next question

### What we learned

- **EAC**: seven of eight members declined; only Somalia improved. Tanzania (−38), South Sudan (−31) and Burundi (−21) fell hardest; Kenya (49) is the regional leader in 2026 — still far below the world mean. The decline was a **political-rights collapse** (PR mean 31.6% → 15.0% of scale; CL 35.6% → 25.4%), and in 2026 **7 of the 8 members are Not Free**.
- **Major powers**: a ~70-point gap separates the democratic powers (81–97) from Russia (12) and China (9). Japan was the only improver; the United States (−12) declined most within the group. **7 Free, 2 Not Free** — and the group's civil-liberties average fell slightly more than political rights.
- **Africa's leaders**: Cabo Verde (92) and Mauritius (87) top the continent; Seychelles (+14) and Malawi (+8) improved. They are the only group whose **political rights rose** (80.2% → 83.0%), their category heatmap is bright (weakest still ~70%), and **all 10 are Free**. Africa's overall mean (38.6) remains far below the world's (56.9).
- **Additivity**: the overall score is **exactly** PR + CL everywhere, and CL = D + E + F + G exactly; PR = A + B + C in ~93% of economy-years (the published PR occasionally deviates). The stacked decompositions show civil liberties dominating every EAC economy's score, and South Sudan's negative PR (−4) preserved in the stack.
- The overall score hides which components moved; the PR/CL split, category heatmaps, status mix and the additive decomposition bring the components into view.

### Next question

*How do broader regions compare?* — notebook 06, regional analysis, extends the grouping approach using the documented UN M49 classifications.